# Explore feature layer item


Explore a feature layer item by visualizing it on a map, examining its schema and attribute values, and generating useful summaries and charts.


## 1. Import libraries and connect to GIS


In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from arcgis.gis import GIS

In [ ]:
gis = GIS("home")

## 2. Get item and view details


In [ ]:
# This value comes from the notebook template configuration.
# Optional: Update the value to run the workflow with a different input.

item_id = "<ITEM-ID>"

In [ ]:
item = gis.content.get(item_id)
item

In [ ]:
print("=== Item Details ===")
print(f"Title    : {item.title}")
print(f"Type     : {item.type}")
print(f"Owner    : {item.owner}")
print(f"Access   : {item.access}")
print(f"Created  : {datetime.fromtimestamp(item.created/1000)}")
print(f"Modified : {datetime.fromtimestamp(item.modified/1000)}")
print(f"URL      : {item.url}")

has_layers = (
    hasattr(item, "layers") and item.layers is not None and len(item.layers) > 0
)
print("\nHas layers:", has_layers)
if has_layers:
    print("Layer count:", len(item.layers))
    for i, lyr in enumerate(item.layers):
        try:
            print(
                f"  [{i}] {lyr.properties.name} (id={lyr.properties.id}, geom={lyr.properties.geometryType})"
            )
        except Exception:
            print(f"  [{i}] <layer>")

## 3. Inspect the layer


### 3.1 Visualize item on a map


In [ ]:
m = gis.map()
m.content.add(item)
m.zoom_to_layer(item)
m

### 3.2 Examine layer fields


Examine field names, data types, aliases, nullability, and any coded value or range domains.


In [ ]:
layer_index = 0  # Optional: change to examine another layer

layer = item.layers[layer_index]

fields_df = pd.DataFrame(layer.properties.fields)
keep_cols = [
    c
    for c in ["name", "alias", "type", "length", "editable", "nullable", "domain"]
    if c in fields_df.columns
]
fields_view = fields_df[keep_cols].copy()

if "domain" in fields_view.columns:
    dom = fields_view["domain"]
    fields_view["domain_type"] = dom.apply(
        lambda d: d.get("type") if isinstance(d, dict) else None
    )
    fields_view["domain_desc"] = dom.apply(
        lambda d: (
            f"CodedValues({len(d.get('codedValues', []))})"
            if isinstance(d, dict) and d.get("type") == "codedValue"
            else (
                f"Range({d.get('minValue')}, {d.get('maxValue')})"
                if isinstance(d, dict) and d.get("type") == "range"
                else None
            )
        )
    )

display(fields_view)

### 3.3 Get feature count and layer extent


In [ ]:
feature_count = layer.query(return_count_only=True)
print(f"Feature count: {feature_count}")

print("\nLayer extent from properties:")
display(layer.properties.extent)

## 4. Review attribute data


### 4.1 Query features


In [ ]:
where = "1=1"  # Optional: narrow the query by updating the WHERE clause

layer = item.layers[layer_index]
query_result = layer.query(where=where)
query_result_df = pd.DataFrame([f.attributes for f in query_result.features])
display(query_result_df.head())

### 4.2 Check data quality


In [ ]:
print("=== Missing values ===")
nulls = query_result_df.isna().sum().sort_values(ascending=False)
display(nulls[nulls > 0])

In [ ]:
print("=== String Hygiene ===")

string_fields = [
    f["name"]
    for f in layer.properties.fields
    if f.get("type") == "esriFieldTypeString"
    and f.get("name") in query_result_df.columns
]
print("\nString fields detected:", len(string_fields))

if len(string_fields) > 0:
    hygiene_rows = []
    for fn in string_fields:
        s = query_result_df[fn].dropna().astype(str)
        hygiene_rows.append(
            {
                "field": fn,
                "missing": int(query_result_df[fn].isna().sum()),
                "empty_or_whitespace": int((s.str.strip() == "").sum()),
                "leading_space": int(s.str.startswith(" ").sum()),
                "trailing_space": int(s.str.endswith(" ").sum()),
                "unique_values": int(s.nunique()),
            }
        )
    hygiene_df = pd.DataFrame(hygiene_rows).sort_values(
        ["empty_or_whitespace", "missing"], ascending=False
    )
    display(hygiene_df)
else:
    print("No string fields available in the result set.")

### 4.3 Profile numeric fields and detect outliers


In [ ]:
numeric_types = {
    "esriFieldTypeInteger",
    "esriFieldTypeSmallInteger",
    "esriFieldTypeDouble",
    "esriFieldTypeSingle",
}
num_fields = [
    f["name"]
    for f in layer.properties.fields
    if f.get("type") in numeric_types and f.get("name") in query_result_df.columns
]
print("Numeric fields:", len(num_fields))

if len(num_fields) > 0:
    prof = query_result_df[num_fields].describe().T
    prof["missing"] = query_result_df[num_fields].isna().sum()
    display(prof.sort_values("missing", ascending=False))

    outlier_rows = []
    for col in num_fields:
        x = query_result_df[col].dropna()
        if len(x) < 10:
            continue
        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        iqr = q3 - q1
        lo = q1 - 1.5 * iqr
        hi = q3 + 1.5 * iqr
        outlier_rows.append(
            {
                "field": col,
                "outliers_count": int(((x < lo) | (x > hi)).sum()),
                "lower_fence": float(lo),
                "upper_fence": float(hi),
            }
        )
    if len(outlier_rows) > 0:
        display(
            pd.DataFrame(outlier_rows).sort_values("outliers_count", ascending=False)
        )
else:
    print("No numeric fields found in the result set.")

### 4.4 Profile categorical values


In [ ]:
cat_types = {"esriFieldTypeString", "esriFieldTypeInteger", "esriFieldTypeSmallInteger"}
cat_fields = [
    f["name"]
    for f in layer.properties.fields
    if f.get("type") in cat_types and f.get("name") in query_result_df.columns
]

TOP_N = 10
max_fields_to_show = 8

print("Categorical-like fields:", len(cat_fields))

shown = 0
for c in cat_fields:
    if shown >= max_fields_to_show:
        break
    vc = query_result_df[c].value_counts(dropna=False).head(TOP_N)
    print(f"\n=== Top values for: {c} ===")
    display(vc.to_frame("count"))
    shown += 1

### 4.5 Profile date fields


In [ ]:
date_fields = [
    f["name"]
    for f in layer.properties.fields
    if f.get("type") == "esriFieldTypeDate" and f.get("name") in query_result_df.columns
]
print("Date fields:", date_fields)

if len(date_fields) > 0:
    for d in date_fields:
        if pd.api.types.is_numeric_dtype(query_result_df[d]):
            query_result_df[d] = pd.to_datetime(
                query_result_df[d], unit="ms", errors="coerce"
            )
        else:
            query_result_df[d] = pd.to_datetime(query_result_df[d], errors="coerce")

    for d in date_fields:
        s = query_result_df[d].dropna()
        if s.empty:
            continue
        print(f"\n=== Date profile: {d} ===")
        print("Min:", s.min())
        print("Max:", s.max())
        by_month = s.dt.to_period("M").value_counts().sort_index()
        display(by_month.to_frame("count"))
else:
    print("No date fields found in the result set.")

## 5. Visualize field values


Generate simple histograms for numeric fields and bar charts for common categorical values.


In [ ]:
numeric_chart_field = num_fields[0] if len(num_fields) > 0 else None
categorical_chart_field = (
    string_fields[0]
    if "string_fields" in globals() and len(string_fields) > 0
    else (cat_fields[0] if len(cat_fields) > 0 else None)
)

print("Numeric chart field:", numeric_chart_field)
print("Categorical chart field:", categorical_chart_field)

if numeric_chart_field is not None:
    x = query_result_df[numeric_chart_field].dropna()
    plt.figure(figsize=(7, 4))
    plt.hist(x, bins=30)
    plt.title(f"Histogram: {numeric_chart_field}")
    plt.xlabel(numeric_chart_field)
    plt.ylabel("count")
    plt.show()

if categorical_chart_field is not None:
    vc = query_result_df[categorical_chart_field].value_counts(dropna=False).head(10)
    plt.figure(figsize=(9, 4))
    vc.plot(kind="bar")
    plt.title(f"Top values: {categorical_chart_field}")
    plt.xlabel(categorical_chart_field)
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()